<a href="https://colab.research.google.com/github/josejohnv04-ux/Steam-Spotlight/blob/main/Steam_Spotlight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### *[MIST 4600] Fall 2026 · GitHub Project*
# **Steam Spotlight**

**My app helps Steam’s store team decide what games should be on the front page by filtering for recent releases (2020–2024) with strong review momentum, identifying titles with the largest player bases, and scoring critic reception using the Gemini LLM API.**

| # | Sub-question | Tools |
|---|---|---|
| 1 | **Recency:** which recent releases (2020–2024) maintain strong active momentum? | pandas |
| 2 | **Players:** of those recent releases, which ones have the largest player bases? | pandas |
| 3 | **Critics:** what do critics say about the shortlisted games (rave, mixed, or weak)? | Gemini LLM API |

**Requirements checklist:** a user and a decision (above) · 3 sub-questions · a function for each sub-question (`recent_momentum`, `filter_high_players`, `make_shortlist`) · a class the app uses (`FrontPagePick`, Cell 5) · an outside tool (Gemini) · runs from top to bottom · a recommendation at the end.

---
## Before you run it

1. Put this notebook and `steam_games.csv` in the same folder in your Google Drive (or keep `steam_games.csv` in the GitHub repo).
2. Add your Gemini key to Colab Secrets (key icon on the left) with the name `GOOGLE_API_KEY`, and turn on **Notebook access**. Get the key at [aistudio.google.com](https://aistudio.google.com) with a personal Gmail account.
3. Runtime → **Run all**. Cell 6 calls Gemini 5 times with a 4-second pause between calls, so it takes about 30 seconds.

## Setup: find the data in Drive

Connects to Google Drive and finds the folder that holds `steam_games.csv`. Cell 1 uses that folder, `FOLDER`, to read the file (with a fallback to GitHub if run outside your Drive).

In [4]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    games = pd.read_csv("https://raw.githubusercontent.com/josejohnv04-ux/Steam-Spotlight/main/steam_games.csv")
    print(f"✅ Loaded {len(games)} games from GitHub")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded 1000 games from GitHub


## Cell 1 · Setup: Gemini API

Connects to Gemini using Colab Secrets and checks the loaded `games` shape. The key is read from Colab Secrets, so it never appears in the code (this repository is public).

In [5]:
from google import genai
from google.colab import userdata

key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=key)
MODEL = "gemini-flash-lite-latest"

print(games.shape)

(1000, 26)


## Cell 2 · Sub-question 1: Recency

`recent_momentum` filters the catalog for games released between `start_year` (2020) and `end_year` (2024), sorts them by total review count (`num_reviews_total`) in descending order, and returns the top `n` (5) titles along with their release year, review volume, and positive review percentage[cite: 17].

In [6]:
def recent_momentum(games, start_year=2020, end_year=2024, n=5):
    recent = games[(games["release_year"] >= start_year) & (games["release_year"] <= end_year)]
    top = recent.sort_values("num_reviews_total", ascending=False)
    return top[["name", "release_year", "num_reviews_total", "pct_pos_total"]].head(n)

print(recent_momentum(games))

                  name  release_year  num_reviews_total  pct_pos_total
9        Apex Legends™          2020             983230             67
10  Black Myth: Wukong          2024             825621             96
12          ELDEN RING          2022             749606             92
14      Cyberpunk 2077          2020             729016             85
15       HELLDIVERS™ 2          2024             716489             76


## Cell 3 · Sub-question 2: Players

The front page needs games with proven audience scale. `filter_high_players` keeps the well-reviewed recent hits (80% positive or more) and checks how many have at least 2,000,000 estimated owners (`owners_low`). The last line shows the average player base across all recent hits to compare. It returns the high-player recent hits, `popular_recent`.

In [7]:
def filter_high_players(games):
    recent = games[(games["release_year"] >= 2020) & (games["release_year"] <= 2024)]
    well_reviewed = recent[recent["pct_pos_total"] >= 80]
    popular_recent = well_reviewed[well_reviewed["owners_low"] >= 2000000]
    print(len(popular_recent))
    print(popular_recent[["name", "owners_low", "pct_pos_total"]].head(5))
    print("Average owners_low across all recent games:", round(recent["owners_low"].mean()))
    return popular_recent

popular_recent = filter_high_players(games)

107
                  name  owners_low  pct_pos_total
10  Black Myth: Wukong    50000000             96
12          ELDEN RING    20000000             92
14      Cyberpunk 2077    20000000             85
16     Baldur's Gate 3    20000000             96
20        Phasmophobia    10000000             96
Average owners_low across all recent games: 2418043


## Cell 4 · Sub-question 3: Critics (Gemini)

`get_critic_tone` sends a game's critic review text (or description if critic quotes are missing) to Gemini and asks how critics received it: **rave** (overwhelmingly positive praise), **mixed** (divided or moderate feedback), or **weak** (mostly negative).

* `time.sleep(4)` waits 4 seconds before each call, to stay under the free tier's limit on requests per minute.
* The answer is cleaned with `.strip().lower().strip(".")`, so `"Rave.\n"` becomes `"rave"`.
* Only the three answers in `TONES` are accepted; anything else becomes `"unknown"`, so one odd answer can't break the results.

In [8]:
TONES = ["rave", "mixed", "weak"]

def get_critic_tone(review_text):
    time.sleep(4)
    if pd.isna(review_text) or str(review_text).strip() == "":
        return "unknown"
    prompt = f"What do critics say about this game? Answer with one word: rave, mixed, or weak. Text: {review_text}"
    response = client.models.generate_content(
        model=MODEL, contents=prompt)
    answer = response.text.strip().lower().strip(".")
    if answer in TONES:
        return answer
    return "unknown"

Try it on one game:

In [9]:
text_col = "reviews" if "reviews" in games.columns else "short_description"
print(games["name"][0])
print(get_critic_tone(games[text_col][0]))

Counter-Strike 2
mixed


## Cell 5 · A class for each pick

Each game on the shortlist becomes a `FrontPagePick` object with its name, price, player count (`owners_low`), and critic tone.

* `good_for_front_page()` returns `True` if critics gave it a `"rave"` tone.
* `card()` returns one line of text for Steam's store team, showing `"free"` for free-to-play titles.

In [10]:
class FrontPagePick:
    def __init__(self, name, price, owners, tone):
        self.name = name
        self.price = price
        self.owners = owners
        self.tone = tone

    def good_for_front_page(self):
        return self.tone == "rave"

    def card(self):
        if self.price == 0:
            cost = "free"
        else:
            cost = f"${self.price:.2f}"
        return f"{self.name}: {cost}, {self.owners:,}+ owners, critics say {self.tone}"

Try the class with a made-up pick:

In [11]:
test = FrontPagePick("Test Game", 59.99, 5000000, "rave")
print(test.card())
print(test.good_for_front_page())

Test Game: $59.99, 5,000,000+ owners, critics say rave
True


## Cell 6 · Sub-question 3 and the shortlist

1. `make_shortlist` takes the top 5 high-player recent hits sorted by total reviews and labels each game's critic reception with `get_critic_tone` (5 Gemini calls, not 1,000, to stay inside the free tier).
2. Each game becomes a `FrontPagePick`; we print its card and list the top front-page picks with rave critic reception.
3. The shortlist is saved to `front_page_picks.csv`.

In [12]:
def make_shortlist(popular_recent):
    text_col = "reviews" if "reviews" in popular_recent.columns else "short_description"
    shortlist = popular_recent.sort_values("num_reviews_total", ascending=False).head(5).copy()
    shortlist["tone"] = shortlist[text_col].apply(get_critic_tone)
    return shortlist

shortlist = make_shortlist(popular_recent)

picks = []
# AI helped: to_dict("records") turns the table into a list of
# dictionaries, one per row (like the Week 7 list of dictionaries)
for game in shortlist.to_dict("records"):
    picks.append(FrontPagePick(game["name"], game["price"], game["owners_low"], game["tone"]))

for pick in picks:
    print(pick.card())

print("\nBest for Steam's front page:")
for pick in picks:
    if pick.good_for_front_page():
        print("-", pick.name)

shortlist.to_csv("front_page_picks.csv", index=False)

Black Myth: Wukong: $59.99, 50,000,000+ owners, critics say unknown
ELDEN RING: $59.99, 20,000,000+ owners, critics say rave
Cyberpunk 2077: $59.99, 20,000,000+ owners, critics say mixed
Baldur's Gate 3: $59.99, 20,000,000+ owners, critics say weak
Phasmophobia: $19.99, 10,000,000+ owners, critics say mixed

Best for Steam's front page:
- ELDEN RING


---
## The recommendation

**Sub-question 1 (Cell 2):** Filtering for recent releases (2020–2024) sorted by total review momentum surfaces top active titles led by *Apex Legends™* (983,230 reviews; 67% positive), *Black Myth: Wukong* (825,621 reviews; 96% positive), *ELDEN RING* (749,606 reviews; 92% positive), *Cyberpunk 2077* (729,016 reviews; 85% positive), and *HELLDIVERS™ 2* (716,489 reviews; 76% positive).

**Sub-question 2 (Cell 3):** Filtering recent releases for at least 80% positive reviews and 2,000,000+ estimated owners yields **107** proven blockbusters—led by *Black Myth: Wukong* (50,000,000+ owners; 96% positive), *ELDEN RING* (20,000,000+ owners; 92% positive), *Cyberpunk 2077* (20,000,000+ owners; 85% positive), *Baldur's Gate 3* (20,000,000+ owners; 96% positive), and *Phasmophobia* (10,000,000+ owners; 96% positive)—far above the 2,418,043 average owner baseline across all recent releases.

**Sub-question 3 (Cell 6):** On the top 5 shortlist, Gemini scores *ELDEN RING* as `"rave"`, *Cyberpunk 2077* and *Phasmophobia* as `"mixed"`, *Baldur's Gate 3* as `"weak"`, and *Black Myth: Wukong* as `"unknown"`, placing **ELDEN RING** alone on the "Best for Steam's front page" list.

**For Steam's store team:** Feature **ELDEN RING** ($59.99, 20,000,000+ owners, 92% positive reviews) in the primary front-page spotlight, as it is the only shortlisted hit combining recent release momentum, massive player scale, and rave critical reception. For secondary banner slots, pair it with **Black Myth: Wukong** or **Baldur's Gate 3** (both holding 96% positive player ratings and tens of millions of owners).